# 04 pretrain — COCO captions, ITC + ITM, asymmetric fusion

**Purpose.** Cross-modal pretraining on COCO 2014 captions for the asymmetric VQA model.
Produces `results/checkpoints/pretrain_final.pt`, which `04_train_clip_upgrade_v2.ipynb`
loads as initialization for VQA finetuning.

**Architecture** (identical to `04_train_clip_upgrade_v2.ipynb`, so the checkpoint is drop-in):

- Vision: open_clip **ViT-L/14 @ 336** with LAION-2B weights (`laion2b_s32b_b82k`). 24×24+1=577 tokens.
- Text: **RoBERTa-large** (1024 hidden), pooler bypassed.
- Fusion: **depth-4 asymmetric stack**. Each block does image-queries-attend-to-text FIRST,
  then text-queries-attend-to-image-attended. No weight sharing across blocks. `EMBED_DIM=768`.
- Pooling: learnable-query attention pool on each stream (shared with finetuning).

**Objectives.**

- **ITC** (image-text contrastive): symmetric InfoNCE over pooled image/text features, projected
  to 256-d and L2-normalized. CLIP-style learnable temperature `logit_scale`, clamped to log(100).
- **ITM** (image-text matching): binary head on `[img_vec; txt_vec]`. Positives are the true pair.
  Hard negatives are mined from the ITC similarity matrix each step (one text-neg per image, one
  image-neg per text). Total ITM CE over `3*B` examples.

**Health gates** (printed each epoch; do NOT use the checkpoint downstream if final gates fail):

| Metric | Epoch 3 | Epoch 6 (final) |
| --- | --- | --- |
| ITC R@1 (image→text) on 5k val | ≥ 35% | ≥ 50% |
| ITM acc on 5k pos + 5k neg | — | ≥ 75% |

If the final gates fail, the cell at the bottom writes `results/PRETRAIN_GATE_FAILED.txt`;
the finetuning notebook reads this marker and falls back to LAION-2B init (run name suffix `_pt0_`)
instead of using a broken pretrained checkpoint.

**No EMA in pretraining** — extra complexity for negligible gain on alignment objectives.
**bf16 autocast on L4 + Ada Lovelace**, no GradScaler.

**Data.** COCO train2014 (~83k images, ~5 captions each, sampled fresh per epoch) for training.
A deterministic seeded 5k slice of val2014 for the gates. Captions zip auto-downloaded; 336-px H5
auto-built if missing.


## Install dependencies

In [ ]:
!pip install -q torch torchvision transformers open_clip_torch matplotlib tqdm Pillow h5py bitsandbytes


## Materialize `models.py` (shared with `04_train_clip_upgrade_v2.ipynb`)

Both notebooks write byte-identical `models.py` into the notebook's working directory.
If you edit one, edit the other — or regenerate via `/tmp/build_notebooks.py`.

In [ ]:
%%writefile models.py
# models.py - shared across 04_pretrain_coco_itc_itm.ipynb and 04_train_clip_upgrade_v2.ipynb.
#
# Both notebooks materialize this exact content via a %%writefile cell so the
# definitions stay in sync. Edit ONE canonical source (this file in the repo
# scratchpad / both notebook cells) and the materialization is reproducible.
#
# The asymmetric cross-modal fusion is the research artifact: each block runs
# image-queries-attend-to-text FIRST, then text-queries-attend-to-the-image-side-
# output. That ordering is preserved across every block in the stack and across
# both notebooks.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip
from transformers import RobertaModel


class CLIPImageEncoder(nn.Module):
    """open_clip ViT-{B-16, L-14} @ configurable image_size, LAION-2B pretrained.
    Returns (B, N_tokens, embed_dim) where N_tokens = (image_size/patch)**2 + 1.

    Manual forward (not visual.output_tokens=True) because that flag drops the
    CLS token; we keep CLS + patches so the fusion sees the full sequence.
    """

    def __init__(self, embed_dim, image_size, model_name, pretrained,
                 freeze=False, grad_checkpointing=False):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(
            model_name, pretrained=pretrained, force_image_size=image_size,
        )
        self.visual = clip_model.visual
        del clip_model
        self.visual.proj = None

        if grad_checkpointing:
            self.visual.set_grad_checkpointing(True)

        if freeze:
            for p in self.visual.parameters():
                p.requires_grad = False

        # Read width dynamically so this works for both ViT-B (768) and ViT-L (1024).
        clip_width = self.visual.transformer.width
        self.projection = nn.Linear(clip_width, embed_dim)

        patch = self.visual.patch_size if isinstance(self.visual.patch_size, int) else self.visual.patch_size[0]
        expected_tokens = (image_size // patch) ** 2 + 1
        actual_tokens = self.visual.positional_embedding.shape[0]
        assert actual_tokens == expected_tokens, (
            f"position embedding interpolation failed: got {actual_tokens} tokens, "
            f"want {expected_tokens} (image_size={image_size}, patch={patch})"
        )
        print(f"[CLIPImageEncoder] {model_name} / {pretrained} @ {image_size} -> "
              f"{actual_tokens} tokens, width={clip_width}, grad_ckpt={grad_checkpointing}")

    def forward(self, images):
        v = self.visual
        x = v.conv1(images)
        x = x.reshape(x.shape[0], x.shape[1], -1)
        x = x.permute(0, 2, 1)

        cls = v.class_embedding.to(x.dtype).view(1, 1, -1).expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + v.positional_embedding.to(x.dtype)
        x = v.patch_dropout(x)
        x = v.ln_pre(x)

        batch_first = getattr(v.transformer, "batch_first", True)
        if not batch_first:
            x = x.permute(1, 0, 2)
        x = v.transformer(x)
        if not batch_first:
            x = x.permute(1, 0, 2)

        x = v.ln_post(x)
        return self.projection(x)


class TextEncoder(nn.Module):
    """RoBERTa text encoder. Reads hidden dim from config (works for both
    roberta-base @ 768 and roberta-large @ 1024). Bypasses the freshly-initialized
    pooler.dense. Calls gradient_checkpointing_enable() if requested.
    """
    POOLING = "last_hidden_state"

    def __init__(self, embed_dim, model_name="roberta-large",
                 freeze=False, grad_checkpointing=False):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        hidden = self.roberta.config.hidden_size
        self.projection = nn.Linear(hidden, embed_dim)

        if grad_checkpointing:
            self.roberta.gradient_checkpointing_enable()
        if freeze:
            for p in self.roberta.parameters():
                p.requires_grad = False

        assert hasattr(self.roberta, "pooler"), \
            "expected RoBERTa to expose a pooler (which we intentionally bypass)"
        print(f"[TextEncoder] {model_name} pooling={self.POOLING} "
              f"hidden_dim={hidden} grad_ckpt={grad_checkpointing}")

    def forward(self, input_ids, attention_mask):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        return self.projection(out.last_hidden_state)


class CrossAttentionBlock(nn.Module):
    """Pre-norm cross-attention block: queries from one modality attend to KVs
    from another, then a residual FFN.

    `return_weights` defaults to False so nn.MultiheadAttention can dispatch to
    the fused scaled_dot_product_attention path (FlashAttention / mem-efficient).
    Pass True only when you actually need the attention map (e.g. visualization).
    """

    def __init__(self, embed_dim, num_heads, dropout=0.1, attn_dropout=None):
        super().__init__()
        if attn_dropout is None:
            attn_dropout = dropout
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=attn_dropout, batch_first=True)
        self.norm_ff = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, query, key_value, key_padding_mask=None, return_weights=False):
        q = self.norm_q(query)
        kv = self.norm_kv(key_value)
        attended, attn_weights = self.cross_attn(
            q, kv, kv, key_padding_mask=key_padding_mask,
            need_weights=return_weights, average_attn_weights=return_weights)
        query = query + attended
        query = query + self.ff(self.norm_ff(query))
        return query, attn_weights  # attn_weights is None when return_weights=False


class AsymmetricCrossModalFusion(nn.Module):
    """Stack of `depth` asymmetric fusion blocks. Each block:
        img_attended = cross_attn_img_to_txt(query=img, kv=txt)
        txt_attended = cross_attn_txt_to_img(query=txt, kv=img_attended)
    Each block has its own learned weights -- no weight sharing across blocks.
    The asymmetric image-first ordering is the research contribution; do not flip.
    """

    def __init__(self, embed_dim, num_heads, depth=1, dropout=0.1, attn_dropout=None):
        super().__init__()
        self.depth = depth
        self.blocks_img_to_txt = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.blocks_txt_to_img = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])

    def forward(self, image_features, text_features, text_padding_mask=None,
                return_weights=False):
        img, txt = image_features, text_features
        last_attn_i2t = last_attn_t2i = None
        for blk_i2t, blk_t2i in zip(self.blocks_img_to_txt, self.blocks_txt_to_img):
            img_attended, last_attn_i2t = blk_i2t(
                query=img, key_value=txt,
                key_padding_mask=text_padding_mask,
                return_weights=return_weights)
            txt_attended, last_attn_t2i = blk_t2i(
                query=txt, key_value=img_attended,
                return_weights=return_weights)
            img, txt = img_attended, txt_attended
        return img, txt, last_attn_i2t, last_attn_t2i


class AttentionPool(nn.Module):
    """Learnable-query attention pool: maps a token sequence to a single vector.
    Supports key_padding_mask for variable-length text streams.
    """

    def __init__(self, embed_dim, num_heads=8):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, key_padding_mask=None):
        q = self.query.expand(x.size(0), -1, -1)
        out, _ = self.attn(q, x, x, key_padding_mask=key_padding_mask,
                           need_weights=False)
        return self.norm(out.squeeze(1))


class AsymmetricVQAModelE2E(nn.Module):
    """End-to-end asymmetric cross-modal model used by BOTH pretraining and finetuning.

    Always constructs encoders + fusion + attention pools.

    If `num_answers` is not None: constructs the VQA classifier.
    If `pretrain_heads=True`: constructs ITC projection MLPs + ITM head + logit_scale.

    Use `pretrain_heads=True, num_answers=None` for pretraining.
    Use `pretrain_heads=False, num_answers=3129` for finetuning.
    `load_state_dict(strict=False)` from a pretrain checkpoint tolerates the
    missing classifier keys and unmatched pretrain-head keys.
    """

    def __init__(self,
                 num_answers,
                 embed_dim=768, num_heads=12, fusion_depth=4,
                 dropout=0.3, attn_dropout=0.2, cls_dropout=0.5,
                 image_size=336,
                 clip_model_name="ViT-L-14", clip_pretrained="laion2b_s32b_b82k",
                 text_model_name="roberta-large",
                 vision_grad_checkpointing=True,
                 text_grad_checkpointing=True,
                 pretrain_heads=False,
                 pool_num_heads=8):
        super().__init__()
        self.image_encoder = CLIPImageEncoder(
            embed_dim=embed_dim, image_size=image_size,
            model_name=clip_model_name, pretrained=clip_pretrained,
            freeze=False, grad_checkpointing=vision_grad_checkpointing,
        )
        self.text_encoder = TextEncoder(
            embed_dim=embed_dim, model_name=text_model_name,
            freeze=False, grad_checkpointing=text_grad_checkpointing,
        )
        self.fusion = AsymmetricCrossModalFusion(
            embed_dim, num_heads, depth=fusion_depth,
            dropout=dropout, attn_dropout=attn_dropout,
        )
        self.pool_img = AttentionPool(embed_dim, num_heads=pool_num_heads)
        self.pool_txt = AttentionPool(embed_dim, num_heads=pool_num_heads)

        self.classifier = None
        if num_answers is not None:
            self.classifier = nn.Sequential(
                nn.Linear(embed_dim * 2, embed_dim),
                nn.ReLU(),
                nn.Dropout(cls_dropout),
                nn.Linear(embed_dim, num_answers),
            )

        self.pretrain_heads = pretrain_heads
        if pretrain_heads:
            self.itc_proj_img = nn.Sequential(
                nn.Linear(embed_dim, embed_dim), nn.GELU(),
                nn.Linear(embed_dim, 256),
            )
            self.itc_proj_txt = nn.Sequential(
                nn.Linear(embed_dim, embed_dim), nn.GELU(),
                nn.Linear(embed_dim, 256),
            )
            self.itm_head = nn.Sequential(
                nn.Linear(2 * embed_dim, embed_dim), nn.GELU(),
                nn.Linear(embed_dim, 2),
            )
            self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def encode_tokens(self, images, input_ids, attention_mask):
        """Encoders only -> (img_tokens, txt_tokens, text_pad_mask). Exposed so
        ITM hard-negative mining can rearrange already-encoded tokens instead of
        re-running the heavy encoders on the mined negatives.
        """
        img_tokens = self.image_encoder(images)
        txt_tokens = self.text_encoder(input_ids, attention_mask)
        text_pad_mask = attention_mask == 0
        return img_tokens, txt_tokens, text_pad_mask

    def fuse_and_pool(self, img_tokens, txt_tokens, text_pad_mask):
        """Fusion + pool on already-encoded tokens -> (img_vec, txt_vec). Used
        by both the positive forward path and the ITM mined-negative path.
        Always runs with return_weights=False so the fused SDPA kernel applies.
        """
        img_att, txt_att, _, _ = self.fusion(
            img_tokens, txt_tokens, text_pad_mask, return_weights=False)
        img_vec = self.pool_img(img_att)
        txt_vec = self.pool_txt(txt_att, key_padding_mask=text_pad_mask)
        return img_vec, txt_vec

    def encode(self, images, input_ids, attention_mask):
        """Encoders + fusion + pool -> (img_vec, txt_vec). Kept for callers that
        don't need the intermediate tokens.
        """
        img_tokens, txt_tokens, text_pad_mask = self.encode_tokens(
            images, input_ids, attention_mask)
        return self.fuse_and_pool(img_tokens, txt_tokens, text_pad_mask)

    def forward(self, images, input_ids, attention_mask, return_attn=False):
        """VQA forward. Set return_attn=True to materialize the fusion attention
        weights (for viz). Default False keeps the fused SDPA kernel hot for
        training / eval / prediction.
        """
        img_tokens, txt_tokens, text_pad_mask = self.encode_tokens(
            images, input_ids, attention_mask)
        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(
            img_tokens, txt_tokens, text_pad_mask, return_weights=return_attn)
        img_vec = self.pool_img(img_att)
        txt_vec = self.pool_txt(txt_att, key_padding_mask=text_pad_mask)
        z = torch.cat([img_vec, txt_vec], dim=-1)
        assert self.classifier is not None, "VQA forward requires num_answers != None"
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}

    def forward_pretrain(self, images, input_ids, attention_mask):
        """Returns {img_vec, txt_vec, img_emb, txt_emb, logit_scale,
                    img_tokens, txt_tokens, text_pad_mask}.
        img_emb / txt_emb are L2-normalized 256-d projections for ITC.
        img_tokens / txt_tokens / text_pad_mask are surfaced so the ITM step can
        rearrange the cached encoder outputs for hard-negative mining without
        re-running the encoders.
        """
        assert self.pretrain_heads, "forward_pretrain requires pretrain_heads=True"
        img_tokens, txt_tokens, text_pad_mask = self.encode_tokens(
            images, input_ids, attention_mask)
        img_vec, txt_vec = self.fuse_and_pool(img_tokens, txt_tokens, text_pad_mask)
        img_proj = self.itc_proj_img(img_vec)
        txt_proj = self.itc_proj_txt(txt_vec)
        img_emb = F.normalize(img_proj, dim=-1)
        txt_emb = F.normalize(txt_proj, dim=-1)
        return {
            "img_vec": img_vec, "txt_vec": txt_vec,
            "img_emb": img_emb, "txt_emb": txt_emb,
            "logit_scale": self.logit_scale,
            "img_tokens": img_tokens, "txt_tokens": txt_tokens,
            "text_pad_mask": text_pad_mask,
        }


## Imports and device

In [ ]:
import json
import math
import random
import re
import shutil
import time
import zipfile
import urllib.request
from collections import Counter
from contextlib import nullcontext
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.amp import autocast
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.transforms import RandAugment
from tqdm import tqdm
from transformers import RobertaTokenizer

# models.py is materialized in the cell above via %%writefile.
import sys
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
from models import (
    CLIPImageEncoder,
    TextEncoder,
    CrossAttentionBlock,
    AsymmetricCrossModalFusion,
    AttentionPool,
    AsymmetricVQAModelE2E,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## Configuration

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# Architecture (must match 04_train_clip_upgrade_v2.ipynb exactly)
CLIP_MODEL          = "ViT-L-14"
CLIP_PRETRAINED     = "laion2b_s32b_b82k"
IMAGE_SIZE          = 336
TEXT_MODEL          = "roberta-large"
EMBED_DIM           = 768
NUM_HEADS           = 12
FUSION_DEPTH        = 4
DROPOUT             = 0.3
ATTN_DROPOUT        = 0.2
USE_GRAD_CHECKPOINT = True

# Pretraining data
MAX_CAPTION_LEN     = 32
VAL_SUBSET_SIZE     = 5000      # held-out slice of COCO val2014 for gates

# Pretraining schedule
BATCH_SIZE          = 32        # no grad accum; ITC wants a single large in-batch group
NUM_WORKERS         = 2
EPOCHS              = 6
ENCODER_LR          = 1e-5
OTHER_LR            = 1e-4
WARMUP_FRAC         = 0.10
WEIGHT_DECAY        = 0.01
GRAD_CLIP_NORM      = 1.0
SEED                = 42

# Smoke flag: True for a cheap dev run (small subset, 1 epoch) to verify the pipeline.
# Flip to False for the real 6-epoch run.
SMOKE_TEST          = True

# Paths
DATA_DIR            = Path("../data")
CHECKPOINT_DIR      = Path("../results/checkpoints")
METRICS_DIR         = Path("../results/metrics")
FIGURES_DIR         = Path("../results/figures")
for d in [CHECKPOINT_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print(f"[Config] arch=CLIP-L/14@336 + RoBERTa-large + asym-fusion d={FUSION_DEPTH} "
      f"embed_dim={EMBED_DIM} | bs={BATCH_SIZE} epochs={EPOCHS} smoke={SMOKE_TEST}")


## Download COCO captions (idempotent)

Pulls `annotations_trainval2014.zip` (~250 MB) from cocodataset.org if `data/captions/captions_{train,val}2014.json` are not already present.

In [ ]:
# Idempotent: download COCO 2014 captions if missing.
CAPTIONS_DIR = Path("../data/captions")
CAPTIONS_DIR.mkdir(parents=True, exist_ok=True)
CAP_TRAIN = CAPTIONS_DIR / "captions_train2014.json"
CAP_VAL   = CAPTIONS_DIR / "captions_val2014.json"

if CAP_TRAIN.exists() and CAP_VAL.exists():
    print(f"[Captions] already present at {CAPTIONS_DIR}")
else:
    zip_path = CAPTIONS_DIR / "annotations_trainval2014.zip"
    url = "http://images.cocodataset.org/annotations/annotations_trainval2014.zip"
    if not zip_path.exists():
        print(f"[Captions] downloading {url} ...")
        urllib.request.urlretrieve(url, zip_path)
        print(f"[Captions] downloaded -> {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)")
    print("[Captions] extracting captions JSONs ...")
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith("captions_train2014.json"):
                with zf.open(name) as src, open(CAP_TRAIN, "wb") as dst:
                    shutil.copyfileobj(src, dst)
            elif name.endswith("captions_val2014.json"):
                with zf.open(name) as src, open(CAP_VAL, "wb") as dst:
                    shutil.copyfileobj(src, dst)
    zip_path.unlink(missing_ok=True)
    print(f"[Captions] extracted to {CAPTIONS_DIR}")

assert CAP_TRAIN.exists() and CAP_VAL.exists(), "captions extraction failed"


## Build the 336-px H5 image cache (idempotent)

About ~40 GB on disk. Takes ~30 min from a cold start. Skips if `data/vqa_images_336.h5` already exists.

In [ ]:
# Idempotent: build vqa_images_336.h5 if missing.
# Same image set as vqa_images_384.h5, just resized to 336x336 (matches ViT-L/14 patch grid 24).
DATA_DIR = Path("../data")
LOCAL_H5_336 = DATA_DIR / "vqa_images_336.h5"
IMAGE_SIZE_H5 = 336

if LOCAL_H5_336.exists():
    print(f"[H5] using existing cache: {LOCAL_H5_336} ({LOCAL_H5_336.stat().st_size / 1e9:.1f} GB)")
else:
    preprocess = transforms.Compose([
        transforms.Resize(IMAGE_SIZE_H5, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMAGE_SIZE_H5),
    ])
    image_dirs = [DATA_DIR / "images" / "train2014", DATA_DIR / "images" / "val2014"]
    all_paths = []
    for d in image_dirs:
        if d.exists():
            all_paths.extend(sorted(d.glob("*.jpg")))
    n = len(all_paths)
    est_gb = (n * IMAGE_SIZE_H5 * IMAGE_SIZE_H5 * 3) / 1e9
    print(f"[H5] building from {n:,} images -> ~{est_gb:.1f} GB at {LOCAL_H5_336}")

    t0 = time.time()
    with h5py.File(LOCAL_H5_336, "w") as h5f:
        imgs_ds = h5f.create_dataset(
            "images", shape=(n, IMAGE_SIZE_H5, IMAGE_SIZE_H5, 3),
            dtype=np.uint8, chunks=(1, IMAGE_SIZE_H5, IMAGE_SIZE_H5, 3),
        )
        ids_ds = h5f.create_dataset("image_ids", shape=(n,), dtype=np.int64)
        for i, p in enumerate(tqdm(all_paths, desc=f"H5@{IMAGE_SIZE_H5}")):
            image_id = int(p.stem.split("_")[-1])
            img = Image.open(p).convert("RGB")
            img = preprocess(img)
            imgs_ds[i] = np.array(img)
            ids_ds[i] = image_id
    print(f"[H5] build elapsed: {time.time() - t0:.0f}s")


## Shared helpers (transforms, normalization, seeding)

These mirror the helpers in `04_train_clip_upgrade_v2.ipynb`. We do NOT augment during pretraining — alignment is the goal.

In [ ]:
# Answer normalization (byte-identical to 03_*), transforms (no HorizontalFlip), CLIP norm.
_ARTICLES = {"a", "an", "the"}
_PUNCT_RE = re.compile(r"[^\w\s\']")
_CONTRACTIONS = {
    "dont": "don't", "doesnt": "doesn't", "didnt": "didn't",
    "isnt": "isn't", "arent": "aren't",
    "wasnt": "wasn't", "werent": "weren't",
    "wont": "won't", "cant": "can't", "couldnt": "couldn't",
    "wouldnt": "wouldn't", "shouldnt": "shouldn't",
    "havent": "haven't", "hasnt": "hasn't", "hadnt": "hadn't",
    "thats": "that's", "whats": "what's", "wheres": "where's",
    "theres": "there's", "heres": "here's", "youre": "you're",
    "theyre": "they're", "weve": "we've", "youve": "you've",
}


def normalize_answer(s: str) -> str:
    s = s.lower().strip()
    s = _PUNCT_RE.sub(" ", s)
    s = " ".join(t for t in s.split() if t not in _ARTICLES)
    return _CONTRACTIONS.get(s, s)


def build_answer_vocab(annotations_file, top_k=3129):
    with open(annotations_file) as f:
        annotations = json.load(f)["annotations"]
    counter = Counter()
    for ann in annotations:
        for a in ann["answers"]:
            counter[normalize_answer(a["answer"])] += 1
    most_common = [ans for ans, _ in counter.most_common(top_k)]
    answer_to_idx = {ans: idx for idx, ans in enumerate(most_common)}
    idx_to_answer = {idx: ans for ans, idx in answer_to_idx.items()}
    return answer_to_idx, idx_to_answer


ANSWER_TYPES = ("yes/no", "number", "other")
ANSWER_TYPE_TO_IDX = {t: i for i, t in enumerate(ANSWER_TYPES)}

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)


class RandAugmentNoGeom(RandAugment):
    """RandAugment without geometric ops (Translate/Shear/Rotate).
    VQA labels frequently depend on spatial position; geometric ops corrupt them.
    """
    _DROP_OPS = {"TranslateX", "TranslateY", "ShearX", "ShearY", "Rotate"}

    def _augmentation_space(self, num_bins, image_size):
        space = super()._augmentation_space(num_bins, image_size)
        return {k: v for k, v in space.items() if k not in self._DROP_OPS}


def get_train_transform():
    """VQA-safe augmentation: NO HorizontalFlip (contradicts RandAugmentNoGeom rationale),
    NO Translate/Shear/Rotate. ColorJitter + photometric RandAugment + RandomErasing.
    """
    return transforms.Compose([
        RandAugmentNoGeom(num_ops=2, magnitude=9,
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.10), ratio=(0.3, 3.3), value=0),
    ])


def get_val_transform():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ])


## COCO captions dataset

In [ ]:
class COCOCaptionsDataset(Dataset):
    """COCO 2014 captions paired with the corresponding 336-px H5 image rows.

    Per-image __getitem__ samples ONE random caption from the ~5 available, so
    re-sampling across epochs gives data diversity for free.

    Returns (image, input_ids, attention_mask, image_id).
    """

    def __init__(self, captions_json, h5_path, tokenizer, max_caption_len=32,
                 transform=None, restrict_image_ids=None):
        self.h5_path = Path(h5_path)
        self.max_caption_len = max_caption_len
        self.transform = transform or get_val_transform()
        self.tokenizer = tokenizer
        self._h5f = None

        with h5py.File(self.h5_path, "r") as f:
            image_ids = f["image_ids"][:]
        self.id_to_row = {int(iid): i for i, iid in enumerate(image_ids)}

        with open(captions_json) as f:
            data = json.load(f)
        captions_by_image = {}
        for ann in data["annotations"]:
            captions_by_image.setdefault(ann["image_id"], []).append(ann["caption"])

        keep_ids = set(restrict_image_ids) if restrict_image_ids is not None else None
        self.records = []
        for image_id, caps in captions_by_image.items():
            if image_id not in self.id_to_row:
                continue
            if keep_ids is not None and image_id not in keep_ids:
                continue
            self.records.append((image_id, caps))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        image_id, caps = self.records[idx]
        if self._h5f is None:
            self._h5f = h5py.File(self.h5_path, "r")
        row = self.id_to_row[image_id]
        img_array = self._h5f["images"][row]
        image = Image.fromarray(img_array)
        image = self.transform(image)

        caption = caps[random.randrange(len(caps))]
        enc = self.tokenizer(
            caption,
            max_length=self.max_caption_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return (
            image,
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            int(image_id),
        )


## Data loaders

In [ ]:
# Single shared tokenizer (per-worker RAM saver).
tokenizer = RobertaTokenizer.from_pretrained(TEXT_MODEL)

# Train: full COCO train2014.
train_ds = COCOCaptionsDataset(
    captions_json=CAP_TRAIN, h5_path=LOCAL_H5_336, tokenizer=tokenizer,
    max_caption_len=MAX_CAPTION_LEN, transform=get_val_transform(),
)

# Val: deterministic seeded 5k slice of val2014.
val_full = COCOCaptionsDataset(
    captions_json=CAP_VAL, h5_path=LOCAL_H5_336, tokenizer=tokenizer,
    max_caption_len=MAX_CAPTION_LEN, transform=get_val_transform(),
)
rng = random.Random(SEED)
val_indices = rng.sample(range(len(val_full)), k=min(VAL_SUBSET_SIZE, len(val_full)))
val_ds = Subset(val_full, val_indices)

if SMOKE_TEST:
    train_ds = Subset(train_ds, list(range(min(2000, len(train_ds)))))
    val_ds   = Subset(val_full, val_indices[:500])
    print(f"[SMOKE] train={len(train_ds)} val={len(val_ds)}")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0, prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0, prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
print(f"Train pairs: {len(train_ds):,} | Val pairs: {len(val_ds):,}")


## Build model (pretrain_heads=True)

In [ ]:
# pretrain_heads=True attaches itc_proj_*, itm_head, logit_scale.
# num_answers=None skips the VQA classifier entirely.
model = AsymmetricVQAModelE2E(
    num_answers=None,
    embed_dim=EMBED_DIM, num_heads=NUM_HEADS, fusion_depth=FUSION_DEPTH,
    dropout=DROPOUT, attn_dropout=ATTN_DROPOUT,
    image_size=IMAGE_SIZE,
    clip_model_name=CLIP_MODEL, clip_pretrained=CLIP_PRETRAINED,
    text_model_name=TEXT_MODEL,
    vision_grad_checkpointing=USE_GRAD_CHECKPOINT,
    text_grad_checkpointing=USE_GRAD_CHECKPOINT,
    pretrain_heads=True,
).to(device)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"[Model] trainable={n_train:,} / total={n_total:,}")

# Sanity forward pass
with torch.no_grad():
    _b = next(iter(val_loader))
    _img, _ids, _mask, _ = [x.to(device) if isinstance(x, torch.Tensor) else x for x in _b]
    with autocast("cuda", dtype=torch.bfloat16):
        _out = model.forward_pretrain(_img, _ids, _mask)
    print(f"[Sanity] img_vec={tuple(_out['img_vec'].shape)} "
          f"txt_vec={tuple(_out['txt_vec'].shape)} "
          f"img_emb={tuple(_out['img_emb'].shape)} "
          f"logit_scale={_out['logit_scale'].item():.3f}")
    assert _out["img_vec"].shape == (_img.size(0), EMBED_DIM)
    assert _out["img_emb"].shape == (_img.size(0), 256)
del _b, _img, _ids, _mask, _out


## Optimizer + scheduler

In [ ]:
# Optimizer: two LRs (encoders vs everything else), no-decay on biases/LayerNorm.
# 8-bit AdamW (bitsandbytes) shrinks optimizer state from ~12 B/param to ~2.5 B/param.
# Scalar logit_scale is forced to fp32 state via GlobalOptimManager, matching the
# CLIP/BLIP convention for the temperature parameter.
import bitsandbytes as bnb

NO_DECAY_KEYS = ("bias", "LayerNorm.weight", "layer_norm.weight", "ln_")

def _split(named):
    decay, nodecay = [], []
    for n, p in named:
        (nodecay if any(k in n for k in NO_DECAY_KEYS) else decay).append(p)
    return decay, nodecay

enc_named   = [(n, p) for n, p in model.named_parameters()
               if p.requires_grad and ("image_encoder" in n or "text_encoder" in n)]
other_named = [(n, p) for n, p in model.named_parameters()
               if p.requires_grad and not ("image_encoder" in n or "text_encoder" in n)]
enc_d, enc_nd   = _split(enc_named)
oth_d, oth_nd   = _split(other_named)

# Keep the scalar temperature in fp32 optimizer state.
bnb.optim.GlobalOptimManager.get_instance().register_module_override(
    model, "logit_scale", {"optim_bits": 32})

optimizer = bnb.optim.AdamW8bit([
    {"params": enc_d,   "lr": ENCODER_LR, "weight_decay": WEIGHT_DECAY},
    {"params": enc_nd,  "lr": ENCODER_LR, "weight_decay": 0.0},
    {"params": oth_d,   "lr": OTHER_LR,   "weight_decay": WEIGHT_DECAY},
    {"params": oth_nd,  "lr": OTHER_LR,   "weight_decay": 0.0},
])

total_steps  = len(train_loader) * EPOCHS
warmup_steps = max(1, int(round(total_steps * WARMUP_FRAC)))

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f"[Opt] AdamW8bit steps={total_steps} warmup={warmup_steps} ({WARMUP_FRAC*100:.0f}%)")


## Training step (ITC + ITM with hard-negative mining)

In [ ]:
def pretrain_one_epoch(model, loader, optimizer, scheduler, log_peak_mem=False):
    model.train()
    n = 0
    tot_itc = 0.0
    tot_itm = 0.0
    peak_logged = not log_peak_mem
    pbar = tqdm(loader, desc="  pretrain", leave=False)
    for step, batch in enumerate(pbar):
        images, ids, masks, _img_ids = batch
        images = images.to(device, non_blocking=True)
        ids    = ids.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)
        B = images.size(0)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", dtype=torch.bfloat16):
            out = model.forward_pretrain(images, ids, masks)
            img_emb = out["img_emb"]; txt_emb = out["txt_emb"]
            img_vec = out["img_vec"]; txt_vec = out["txt_vec"]
            img_tokens = out["img_tokens"]; txt_tokens = out["txt_tokens"]
            text_pad_mask = out["text_pad_mask"]
            logit_scale = out["logit_scale"].exp()

            # --- ITC: symmetric InfoNCE
            sim = logit_scale * (img_emb @ txt_emb.T)            # (B, B)
            tgt = torch.arange(B, device=device)
            loss_itc = 0.5 * (F.cross_entropy(sim, tgt) + F.cross_entropy(sim.T, tgt))

            # --- ITM: hard-negative mining from sim (detached)
            with torch.no_grad():
                sim_det = sim.detach().clone()
                sim_det.fill_diagonal_(float("-inf"))
                txt_neg_idx = sim_det.argmax(dim=1)              # for each image, hardest non-matching text
                img_neg_idx = sim_det.argmax(dim=0)              # for each text,  hardest non-matching image

            # Reindex cached encoder tokens for the mined pairings; only fusion+pool
            # re-runs. Gradients still flow through the encoders via img_tokens /
            # txt_tokens (they are not detached), so this is mathematically the
            # same mined-neg ITM as the older "model.encode(...)" path, minus two
            # full encoder forwards per step.
            img_vec_with_neg_txt, _ = model.fuse_and_pool(
                img_tokens, txt_tokens[txt_neg_idx], text_pad_mask[txt_neg_idx])
            _, txt_vec_with_neg_img = model.fuse_and_pool(
                img_tokens[img_neg_idx], txt_tokens, text_pad_mask)

            # Three categories: positive, text-negative, image-negative.
            itm_inputs = torch.cat([
                torch.cat([img_vec,             txt_vec],              dim=-1),
                torch.cat([img_vec_with_neg_txt, txt_vec[txt_neg_idx]], dim=-1),
                torch.cat([img_vec[img_neg_idx], txt_vec_with_neg_img], dim=-1),
            ], dim=0)
            itm_labels = torch.cat([
                torch.ones(B,  device=device, dtype=torch.long),
                torch.zeros(B, device=device, dtype=torch.long),
                torch.zeros(B, device=device, dtype=torch.long),
            ], dim=0)
            itm_logits = model.itm_head(itm_inputs)
            loss_itm = F.cross_entropy(itm_logits, itm_labels)

            loss = loss_itc + loss_itm

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        scheduler.step()
        # Keep logit_scale numerically stable.
        with torch.no_grad():
            model.logit_scale.clamp_(0.0, math.log(100.0))

        n += B
        tot_itc += loss_itc.item() * B
        tot_itm += loss_itm.item() * B
        pbar.set_postfix(itc=loss_itc.item(), itm=loss_itm.item(),
                         ls=model.logit_scale.exp().item())

        if not peak_logged and torch.cuda.is_available():
            peak_gb = torch.cuda.max_memory_allocated() / 1e9
            print(f"[Mem] peak after first optimizer step: {peak_gb:.2f} GB "
                  f"(bs={BATCH_SIZE}, ckpt={USE_GRAD_CHECKPOINT})")
            peak_logged = True

    return {"train_loss_itc": tot_itc / max(1, n),
            "train_loss_itm": tot_itm / max(1, n),
            "train_loss": (tot_itc + tot_itm) / max(1, n)}


## Evaluation: ITC R@1 + ITM accuracy

In [ ]:
@torch.no_grad()
def evaluate_pretrain(model, loader):
    """Compute ITC R@1 (image->text) over the entire val slice + ITM accuracy on
    a balanced pos / mined-neg set built from the same slice."""
    model.eval()
    all_img, all_txt = [], []
    all_img_vec, all_txt_vec = [], []
    for batch in tqdm(loader, desc="  eval", leave=False):
        images, ids, masks, _img_ids = batch
        images = images.to(device, non_blocking=True)
        ids    = ids.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)
        with autocast("cuda", dtype=torch.bfloat16):
            out = model.forward_pretrain(images, ids, masks)
        all_img.append(out["img_emb"].float().cpu())
        all_txt.append(out["txt_emb"].float().cpu())
        all_img_vec.append(out["img_vec"].float().cpu())
        all_txt_vec.append(out["txt_vec"].float().cpu())
    img_emb = torch.cat(all_img)
    txt_emb = torch.cat(all_txt)
    img_vec = torch.cat(all_img_vec)
    txt_vec = torch.cat(all_txt_vec)
    N = img_emb.size(0)

    # ITC R@1: per image, argmax similarity over all N texts; correct iff diagonal wins.
    sim = img_emb @ txt_emb.T
    preds = sim.argmax(dim=1)
    r1 = (preds == torch.arange(N)).float().mean().item() * 100

    # ITM accuracy on N positives + N hard-mined negatives (text-side only, for speed).
    sim_det = sim.clone()
    sim_det.fill_diagonal_(float("-inf"))
    txt_neg_idx = sim_det.argmax(dim=1)

    pos_in = torch.cat([img_vec, txt_vec], dim=-1).to(device)
    neg_in = torch.cat([img_vec, txt_vec[txt_neg_idx]], dim=-1).to(device)
    inputs = torch.cat([pos_in, neg_in], dim=0)
    labels = torch.cat([torch.ones(N, dtype=torch.long), torch.zeros(N, dtype=torch.long)]).to(device)
    with autocast("cuda", dtype=torch.bfloat16):
        logits = model.itm_head(inputs)
    acc = (logits.argmax(dim=1) == labels).float().mean().item() * 100

    return {"val_itc_r1": r1, "val_itm_acc": acc}


## Training loop

In [ ]:
history = []
best_r1 = -1.0
for epoch in range(1, EPOCHS + 1):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_m = pretrain_one_epoch(
        model, train_loader, optimizer, scheduler, log_peak_mem=(epoch == 1),
    )
    val_m = evaluate_pretrain(model, val_loader)
    elapsed = time.time() - t0
    peak_gb = (torch.cuda.max_memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
    epoch_data = {
        "epoch": epoch,
        **train_m,
        **val_m,
        "peak_mem_gb": round(peak_gb, 2),
        "elapsed_s": round(elapsed, 1),
    }
    history.append(epoch_data)
    print(f"  Epoch {epoch}/{EPOCHS} | itc {train_m['train_loss_itc']:.3f} "
          f"itm {train_m['train_loss_itm']:.3f} | "
          f"[ITC] epoch {epoch} R@1={val_m['val_itc_r1']:.2f}% | "
          f"[ITM] epoch {epoch} acc={val_m['val_itm_acc']:.2f}% | "
          f"peak={peak_gb:.1f}GB | {elapsed:.0f}s")

    # Save per-epoch (model weights only — small enough to keep all 6).
    ckpt_path = CHECKPOINT_DIR / f"pretrain_epoch{epoch}.pt"
    torch.save({"model": model.state_dict(), "metrics": epoch_data}, ckpt_path)

    # Track best by R@1 (not required for gate logic, but useful diagnostic).
    if val_m["val_itc_r1"] > best_r1:
        best_r1 = val_m["val_itc_r1"]

    with open(METRICS_DIR / "pretrain_history.json", "w") as f:
        json.dump(history, f, indent=2)

# Final checkpoint — this is the artifact the finetuning notebook consumes.
torch.save({"model": model.state_dict(), "metrics": history[-1]},
           CHECKPOINT_DIR / "pretrain_final.pt")
print(f"\nPretraining complete. Best R@1: {best_r1:.2f}%. Saved pretrain_final.pt.")


## Pretraining curves

In [ ]:
if history:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = [h["epoch"] for h in history]
    axes[0].plot(epochs, [h["train_loss_itc"] for h in history], label="ITC loss")
    axes[0].plot(epochs, [h["train_loss_itm"] for h in history], label="ITM loss")
    axes[0].set(xlabel="Epoch", ylabel="Loss", title="Pretraining train loss")
    axes[0].legend()

    axes[1].plot(epochs, [h["val_itc_r1"] for h in history], "o-", label="ITC R@1 (image->text)")
    axes[1].plot(epochs, [h["val_itm_acc"] for h in history], "s-", label="ITM acc")
    axes[1].axhline(50, color="gray", linestyle="--", alpha=0.5, label="R@1 gate (epoch 6)")
    axes[1].axhline(75, color="gray", linestyle=":",  alpha=0.5, label="ITM gate (epoch 6)")
    axes[1].set(xlabel="Epoch", ylabel="%", title="Pretraining val gates")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "pretrain_curves.png", dpi=150, bbox_inches="tight")
    plt.show()


## Final gate check

Writes `results/PRETRAIN_GATE_FAILED.txt` if the final gates are not met. The finetuning notebook reads this marker.

In [ ]:
# Final gate check. If the thresholds are not met, write a marker file so
# the finetuning notebook can fall back to LAION-2B init and tag the run name pt0.
GATE_MARKER = CHECKPOINT_DIR.parent / "PRETRAIN_GATE_FAILED.txt"
final = history[-1] if history else None
if final is None:
    print("[GATE] no history — nothing to check.")
elif SMOKE_TEST:
    print(f"[GATE] SMOKE_TEST=True; skipping hard gates. final R@1={final['val_itc_r1']:.2f}% "
          f"ITM={final['val_itm_acc']:.2f}%")
    GATE_MARKER.unlink(missing_ok=True)
else:
    r1, itm = final["val_itc_r1"], final["val_itm_acc"]
    ok = (r1 >= 50.0) and (itm >= 75.0)
    if ok:
        print(f"[GATE PASS] R@1={r1:.2f}% (>=50%) ITM={itm:.2f}% (>=75%). Proceed to finetuning.")
        GATE_MARKER.unlink(missing_ok=True)
    else:
        msg = (f"R@1={r1:.2f}% (need >=50%), ITM={itm:.2f}% (need >=75%). "
               f"Finetuning will fall back to LAION-2B init.")
        print(f"[GATE FAIL] {msg}")
        GATE_MARKER.write_text(msg + "\n")
